In [ ]:
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import numpy as np
import warnings
warnings.simplefilter(action='ignore')

## Lecture 12 ##

## A Join Example ##

In [ ]:
full = Table.read_table('nc-est2019-agesex-res.csv')
census = full.select('SEX', 'AGE', 'POPESTIMATE2019')
census.show(3)

In [ ]:
sex_codes = Table().with_columns(
    'SEX CODE', make_array(0, 1, 2),
    'CODE DEFINITION', make_array('All', 'Selected Male', 'Selected Female')
)
sex_codes

## E-Scooter Trips in Minneapolis, July 2019 ##

In [ ]:
trips = Table.read_table('EScooterTrips_Jul2019.csv')
trips

## Distribution of Durations ##

In [ ]:
trips.hist('TripDuration')
plt.show();

In [ ]:
# Approx percent of people who have 
# a ride duration between 200 and 400 seconds
# "between" = [200, 400) 


## Start and End Points ##

In [ ]:
# Most common start point


## Fastest Trips between Locations ##

How can we find the fastest trip ever between each pair of addresses?

## Practice question

Find the 5 locations closest to 0 WASHINGTON AVE S by minimum trip time.

## Maps ##

In [ ]:
geo_data = Table.read_table('mpls_scooter_addresses_with_lat_long.csv')
geo_data

In [ ]:
# you don't need to know the details of this cell - the mapping module needs the data in a very particular format
map_data = geo_data.where('Latitude', are.between(40, 48)).where('Longitude', are.between(-99, -70)) # clean out the nans
map_data = map_data.relabeled('Latitude', 'lat').relabeled('Longitude', 'long').relabeled('Address', 'labels') # rename columns to what the mapping module expects
map_data = map_data.select('lat', 'long', 'labels') # only keep the columns the mapping module needs

### Practice question

Map all locations within 4 minutes (minimum ride time) of 0 WASHINGTON AVE S.

In [ ]:
# get rid of the outliers - I happen to know that some of these are just errors/idiosyncrasies in the address data
# from the city
addresses_to_remove = make_array('0 5TH AVE S','0 PORTLAND AVE','0 CHICAGO AVE', '0 4TH AVE S', '0 3RD AVE S',
                                 '0 CHICAGO AVE', '0 10TH AVE S', '300 WASHINGTON AVE N', '200 WASHINGTON AVE N',
                                 '0 WASHINGTON AVE N')
close_map_data = close_map_data.where('labels', are.not_contained_in(addresses_to_remove))
close_map_data

Choose marker colors by the minimum time from Washington Ave

In [ ]:
colors = Table().with_columns(
    "minutes", np.arange(15),
    "colors",  ["darkblue", "blue", "lightblue", 
                "darkgreen", "green", "lightgreen",
                "orange", "darkred", "red",
                "gray", "gray", "gray", 
                "gray", "gray", "gray"])
colors_wash = (from_washington_ave_s
 .with_column("Minutes", minutes)
 .join("Minutes", colors, "minutes"))

colored_markers = (map_data
      .join('labels', colors_wash, 'EndAddress')
      .select('lat', 'long', 'labels', 'colors'))
Marker.map_table(colored_markers) # more mapping magic